In [1]:
!pip install xlogit

In [3]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [5]:
import pandas as pd
import numpy as np
df = pd.read_csv("https://engineering.purdue.edu/~flm/StatEcon-Files/Ex13-1.txt",
                 sep="\t", header=None, prefix="x")
df.rename(columns={'x0': 'choice', 'x6': 'dist', 'x10': 'male', 'x14': 'vehmodel'},
          inplace=True)  # Rename columns of interest
df['alt'] = np.tile(['arterial', 'rural', 'freeway'], len(df)//3)  # Add column with alternatives
df['ids'] = np.repeat(np.arange(len(df)//3), 3)  # Add column with unique ids
df['vehage'] = 86 - df['vehmodel']
df

,choice,x1,x2,x3,x4,x5,dist,x7,x8,x9,male,x11,x12,x13,vehmodel,x15,x16,alt,ids,vehage
0,1,1,0,0,460,14,48,0,0,2,0,1,0,1,86,0,28,arterial,0,0
1,0,0,1,0,440,7,44,0,0,2,0,1,0,1,86,0,28,rural,0,0
2,0,0,0,1,130,7,61,0,0,2,0,1,0,1,86,0,28,freeway,0,0
3,1,1,0,0,595,13,59,1,0,2,1,1,0,2,85,0,27,arterial,1,1
4,0,0,1,0,515,13,70,1,0,2,1,1,0,2,85,0,27,rural,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
448,0,0,1,0,325,10,70,0,1,2,1,1,0,1,84,1,24,rural,149,2
449,1,0,0,1,200,5,74,0,1,2,1,1,0,1,84,1,24,freeway,149,2
450,0,1,0,0,900,14,51,1,0,2,1,1,0,3,83,1,18,arterial,150,3
451,0,0,1,0,550,7,47,1,0,2,1,1,0,3,83,1,18,rural,150,3


In [6]:
# Alternative specific constants
df['asc_rural'] = np.ones(len(df)) * (df['alt'] == 'rural')
df['asc_freeway'] = np.ones(len(df)) * (df['alt'] == 'freeway')

# Distance
df['dist_arterial'] = df['dist'] * (df['alt'] == 'arterial')
df['dist_rural'] = df['dist'] * (df['alt'] == 'rural')
df['dist_freeway'] = df['dist'] * (df['alt'] == 'freeway')

# Vehicle age
df['vehage_rural'] = df['vehage'] * (df['alt'] == 'rural')
df['vehage_freeway'] = df['vehage'] * (df['alt'] == 'freeway')

# Male driver
df['male_freeway'] = df['male'] * (df['alt'] == 'freeway')

In [9]:
from xlogit import MultinomialLogit
varnames=['asc_rural', 'asc_freeway', 'dist_arterial', 'dist_rural',
          'dist_freeway', 'vehage_rural', 'vehage_freeway', 'male_freeway']
model = MultinomialLogit()
model.fit(X=df[varnames], y=df['choice'], varnames=varnames,
          ids=df['ids'], alts=df['alt'])
model.summary()

Optimization terminated successfully.
    Message: The gradients are close to zero
    Iterations: 11
    Function evaluations: 12
Estimation time= 0.0 seconds
---------------------------------------------------------------------------
Coefficient              Estimate      Std.Err.         z-val         P>|z|
---------------------------------------------------------------------------
asc_rural               2.8134599     1.3993600     2.0105333        0.0462 *  
asc_freeway            -2.6865956     2.7277771    -0.9849029         0.326    
dist_arterial          -0.1229127     0.0301181    -4.0810198      7.24e-05 ***
dist_rural             -0.1773682     0.0306588    -5.7852265      4.04e-08 ***
dist_freeway           -0.0956479     0.0473573    -2.0197065        0.0452 *  
vehage_rural            0.1236833     0.0686411     1.8018844        0.0736 .  
vehage_freeway          0.2268707     0.0845617     2.6829004       0.00811 ** 
male_freeway            0.5991537     0.6609786     